# Descargador de Documentos - Municipalidad de Rosario

Este notebook descarga los documentos listados en los CSVs de datos abiertos y los guarda directamente en tu Google Drive.

**Pasos:**
1. Ejecutar la celda de montado de Drive
2. Subir los CSVs del scrapper a la carpeta configurada
3. Ejecutar el resto de las celdas

> **Tip:** Para evitar que la sesión expire en Colab gratuito, conectate con Colab Pro o usá la extensión anti-idle de Chrome.

In [ ]:
# ── 1. Montar Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Configurá tu carpeta base en Drive
DRIVE_BASE = '/content/drive/MyDrive/Rosario_Docs'
SCRAPPER_DIR = '/content/drive/MyDrive/Rosario_Docs/Scrapper'  # donde subiste los CSVs

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print('Drive montado OK. Carpeta base:', DRIVE_BASE)

In [ ]:
# ── 2. Instalar dependencias ────────────────────────────────────────
!pip install -q aiofiles aiohttp tqdm beautifulsoup4 lxml

In [ ]:
# ── 3. Descargar el script desde GitHub (o subilo manualmente) ──────
# Opción A: desde GitHub (si subiste el repo)
# !wget -q https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/downloader.py

# Opción B: pegar el script directamente (se genera el archivo)
script_code = '''
# (pegar aquí el contenido completo de downloader.py)
'''

# Opción C: si ya subiste downloader.py a Drive
import shutil
shutil.copy(f'{DRIVE_BASE}/downloader.py', '/content/downloader.py')
print('Script copiado OK')

In [ ]:
# ── 4. Configurar rutas y correr el descargador ─────────────────────
import subprocess, sys

OUTPUT_DIR   = f'{DRIVE_BASE}/downloads'
CONCURRENCY  = 6    # simultaneous downloads (bajar si hay rate-limit)
DELAY        = 0.5  # segundos entre requests

# Sobreescribir SCRAPPER_DIR dentro del script con un parche inline
# (más simple: pasar variables de entorno)
import os
env = os.environ.copy()
env['SCRAPPER_DIR'] = SCRAPPER_DIR

cmd = [
    sys.executable, '/content/downloader.py',
    '--output', OUTPUT_DIR,
    '--concurrency', str(CONCURRENCY),
    '--delay', str(DELAY),
]
print('Ejecutando:', ' '.join(cmd))
subprocess.run(cmd, env=env)

In [ ]:
# ── 5. Resumen de archivos descargados ─────────────────────────────
from pathlib import Path

base = Path(OUTPUT_DIR)
for folder in sorted(base.iterdir()):
    if folder.is_dir():
        count = len(list(folder.glob('*.pdf')))
        size_mb = sum(f.stat().st_size for f in folder.glob('*.pdf')) / 1_048_576
        print(f'{folder.name:45s} {count:5d} archivos  {size_mb:8.1f} MB')